# 🔍 GridSearchCV & RandomizedSearchCV — Hyperparameter Tuning

> **Folder:** `07_Hyperparameter_Tuning`  
> **Notebook:** `gridsearchcv.ipynb`  
> **Author:** Hamna Munir

---

## 🎯 Objectives

By the end of this notebook, you will:

- Understand **when and why** to tune hyperparameters
- Run **GridSearchCV** — exhaustive search over a parameter grid
- Run **RandomizedSearchCV** — efficient random sampling
- Compare **Grid vs Random** search on the same problem
- Tune multiple models and build a **leaderboard**
- Use **nested CV** to get unbiased performance estimates
- Tune hyperparameters correctly inside a **Pipeline**
- Visualize **heatmaps** and **parallel coordinate plots** of results

---

## 📚 Techniques Covered

| # | Technique | Key Insight |
|---|-----------|-------------|
| 1 | Dataset Setup | Classification + Regression |
| 2 | Manual Baseline | No tuning — default params |
| 3 | GridSearchCV | Exhaustive grid over all combinations |
| 4 | RandomizedSearchCV | Random sampling — faster for large spaces |
| 5 | Grid vs Random Comparison | Same budget, different strategies |
| 6 | Multi-Model Tuning | Tune RF, GBM, SVM, KNN |
| 7 | Heatmap Visualization | 2D param interaction plots |
| 8 | Pipeline + GridSearchCV | Correct leakage-free tuning |
| 9 | Nested CV | Unbiased evaluation after tuning |
| 10 | Regression Tuning | GridSearchCV for Ridge, RF regressor |
| 11 | Summary & Golden Rules | Key takeaways |


---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    GridSearchCV, RandomizedSearchCV,
    cross_val_score, cross_validate,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    r2_score, mean_squared_error,
)
from scipy.stats import randint, uniform, loguniform

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E84855',
    'accent'   : '#3BB273',
    'warning'  : '#F18F01',
    'purple'   : '#7B2D8B',
    'palette'  : ['#2E86AB','#E84855','#3BB273','#F18F01','#7B2D8B','#F4D35E'],
}
print('✅ Libraries loaded successfully!')

---
## 1️⃣ Dataset Setup

> Two datasets for tuning demonstrations:
> - **Binary classification** (balanced) — primary dataset
> - **Regression** — for regression tuning demo


In [ ]:
np.random.seed(42)

# ── Binary classification ─────────────────────────────────────────────────
X_clf, y_clf = make_classification(
    n_samples=800, n_features=20, n_informative=10,
    n_redundant=5, n_classes=2, weights=[0.5, 0.5],
    random_state=42
)
feat_names = [f'F{i+1:02d}' for i in range(20)]
X_clf_df   = pd.DataFrame(X_clf, columns=feat_names)
y_clf_s    = pd.Series(y_clf, name='Target')

# ── Regression ────────────────────────────────────────────────────────────
X_reg, y_reg = make_regression(
    n_samples=600, n_features=15, n_informative=8,
    noise=20, random_state=42
)
X_reg_df = pd.DataFrame(X_reg, columns=[f'R{i+1:02d}' for i in range(15)])
y_reg_s  = pd.Series(y_reg, name='Target')

# ── Scale + split ─────────────────────────────────────────────────────────
sc_clf = StandardScaler()
sc_reg = StandardScaler()

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_clf_df, y_clf_s, test_size=0.2, stratify=y_clf_s, random_state=42)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_reg_df, y_reg_s, test_size=0.2, random_state=42)

Xc_tr_sc = sc_clf.fit_transform(Xc_tr)
Xc_te_sc = sc_clf.transform(Xc_te)
Xr_tr_sc = sc_reg.fit_transform(Xr_tr)
Xr_te_sc = sc_reg.transform(Xr_te)

print(f'Classification : {X_clf_df.shape} | classes={dict(y_clf_s.value_counts().sort_index())}')
print(f'  Train={Xc_tr.shape}  Test={Xc_te.shape}')
print(f'Regression     : {X_reg_df.shape}')
print(f'  Train={Xr_tr.shape}  Test={Xr_te.shape}')

---
## 2️⃣ Manual Baseline — Default Hyperparameters

> Before tuning, always establish a **baseline** with default hyperparameters.  
> This tells you how much tuning actually helps.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_models = {
    'LogisticRegression (default)': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest (default)'      : RandomForestClassifier(random_state=42),
    'GradientBoosting (default)'  : GradientBoostingClassifier(random_state=42),
    'SVM (default)'               : SVC(probability=True, random_state=42),
    'KNN (default)'               : KNeighborsClassifier(),
}

baseline_rows = []
print('Baseline CV Scores (default hyperparameters, 5-Fold):')
for name, model in baseline_models.items():
    scores = cross_val_score(model, Xc_tr_sc, yc_tr, cv=skf,
                              scoring='roc_auc', n_jobs=-1)
    baseline_rows.append({
        'Model'     : name,
        'CV AUC'    : round(scores.mean(), 4),
        'Std'       : round(scores.std(), 4),
    })
    print(f'  {name:40s}: {scores.mean():.4f} ± {scores.std():.4f}')

baseline_df = pd.DataFrame(baseline_rows)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(baseline_df['Model'], baseline_df['CV AUC'],
               xerr=baseline_df['Std'], color=COLORS['primary'],
               alpha=0.80, edgecolor='white', capsize=5)
for bar, val in zip(bars, baseline_df['CV AUC']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
ax.set_xlabel('CV ROC-AUC', fontsize=11)
ax.set_title('Baseline — Default Hyperparameters (5-Fold CV)',
             fontsize=13, fontweight='bold')
ax.set_xlim([0.5, 1.05])
plt.tight_layout()
plt.show()

---
## 3️⃣ GridSearchCV — Exhaustive Search

> GridSearchCV tries **every combination** of the specified hyperparameter values.
>
> ```
> param_grid = {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1]}
> → Evaluates: (0.1, 0.01), (0.1, 0.1), (1, 0.01), (1, 0.1),
>              (10, 0.01), (10, 0.1)  → 6 combinations × 5 folds = 30 fits
> ```
>
> **Total fits = Π(len(values)) × n_splits**  
> ⚠️ Grows exponentially with more parameters — use Random Search for large spaces.


In [ ]:
# ── GridSearchCV on Random Forest ────────────────────────────────────────
param_grid_rf = {
    'n_estimators' : [50, 100, 200],
    'max_depth'    : [3, 5, 7, None],
    'max_features' : ['sqrt', 'log2'],
    'min_samples_leaf': [1, 2, 4],
}

total_combos = 1
for v in param_grid_rf.values():
    total_combos *= len(v)
print(f'Total combinations: {total_combos}')
print(f'Total fits (5-fold): {total_combos * 5}')

gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
)
gs_rf.fit(Xc_tr_sc, yc_tr)

print(f'\nGridSearchCV — Random Forest')
print(f'  Best params : {gs_rf.best_params_}')
print(f'  Best CV AUC : {gs_rf.best_score_:.4f}')
print(f'  Test AUC    : {roc_auc_score(yc_te, gs_rf.predict_proba(Xc_te_sc)[:,1]):.4f}')

# CV results DataFrame
cv_res = pd.DataFrame(gs_rf.cv_results_)
top10  = cv_res.nlargest(10, 'mean_test_score')[
    ['param_n_estimators','param_max_depth','param_max_features',
     'param_min_samples_leaf','mean_test_score','std_test_score',
     'mean_train_score']
].round(4)
top10.columns = ['n_est','max_depth','max_feat','min_leaf',
                 'Val AUC','Val Std','Train AUC']
print('\nTop 10 Configurations:')
print(top10.to_string(index=False))

---
## 4️⃣ RandomizedSearchCV — Efficient Sampling

> RandomizedSearchCV **randomly samples** n_iter configurations from the  
> search space — much faster than GridSearch for large spaces.
>
> Key advantages:
> - Can use **continuous distributions** (not just discrete lists)
> - Same budget explores more of the space than Grid Search
> - Bergstra & Bengio (2012): random beats grid when few params truly matter
>
> ```python
> from scipy.stats import randint, uniform, loguniform
>
> param_dist = {
>     'learning_rate': loguniform(0.001, 0.3),  # continuous log-uniform
>     'n_estimators' : randint(50, 500),         # discrete uniform
>     'subsample'    : uniform(0.5, 0.5),        # continuous uniform [0.5, 1.0]
> }
> ```


In [ ]:
# ── RandomizedSearchCV on GradientBoosting ───────────────────────────────
param_dist_gb = {
    'n_estimators'    : randint(50, 400),
    'max_depth'       : randint(2, 10),
    'learning_rate'   : loguniform(0.005, 0.3),
    'subsample'       : uniform(0.5, 0.5),
    'min_samples_leaf': randint(1, 20),
    'max_features'    : uniform(0.3, 0.7),
}

n_iter = 60
rs_gb = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=n_iter,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=0,
    return_train_score=True,
)
rs_gb.fit(Xc_tr_sc, yc_tr)

print(f'RandomizedSearchCV — GradientBoosting (n_iter={n_iter})')
print(f'  Best params : {rs_gb.best_params_}')
print(f'  Best CV AUC : {rs_gb.best_score_:.4f}')
print(f'  Test AUC    : {roc_auc_score(yc_te, rs_gb.predict_proba(Xc_te_sc)[:,1]):.4f}')

# Convergence plot — best score found vs iteration
rs_scores = pd.DataFrame(rs_gb.cv_results_)['mean_test_score'].values
best_so_far = np.maximum.accumulate(rs_scores)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(range(1, n_iter+1), rs_scores, 'o',
             color=COLORS['primary'], alpha=0.4, markersize=5, label='Each trial')
axes[0].plot(range(1, n_iter+1), best_so_far, '-',
             color=COLORS['secondary'], linewidth=2.5, label='Best so far')
axes[0].axhline(rs_gb.best_score_, color=COLORS['accent'],
                linestyle='--', linewidth=2,
                label=f'Final best={rs_gb.best_score_:.4f}')
axes[0].set_xlabel('Iteration', fontsize=11)
axes[0].set_ylabel('CV ROC-AUC', fontsize=11)
axes[0].set_title('RandomizedSearch — Convergence Plot', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# Distribution of sampled learning rates
lr_vals = [rs_gb.cv_results_[f'param_learning_rate'][i]
           for i in range(n_iter)]
axes[1].scatter(lr_vals, rs_scores,
                c=rs_scores, cmap='RdYlGn', s=60, alpha=0.8, edgecolors='white')
axes[1].set_xscale('log')
axes[1].set_xlabel('Sampled learning_rate (log scale)', fontsize=11)
axes[1].set_ylabel('CV ROC-AUC', fontsize=11)
axes[1].set_title('Score vs learning_rate — Sampled Configurations',
                  fontsize=12, fontweight='bold')

plt.suptitle('RandomizedSearchCV — GradientBoosting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5️⃣ Grid vs Random Search — Same Budget Comparison

> Comparing Grid and Random Search with the **same number of total fits**  
> reveals which is more sample-efficient.
>
> Key insight: Random Search explores a larger region of the search space  
> with the same number of evaluations.


In [ ]:
# Fixed budget: 60 evaluations
budget = 60

# Grid: 3×4×5 = 60 exactly
param_grid_fixed = {
    'n_estimators' : [50, 100, 200],
    'max_depth'    : [2, 4, 6, None],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
}

gs_fixed = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid_fixed, cv=skf, scoring='roc_auc',
    n_jobs=-1, return_train_score=True
)
gs_fixed.fit(Xc_tr_sc, yc_tr)

# Random: same n_iter=60
param_dist_fixed = {
    'n_estimators' : randint(50, 300),
    'max_depth'    : randint(2, 12),
    'learning_rate': loguniform(0.005, 0.3),
}

rs_fixed = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_dist_fixed, n_iter=budget, cv=skf,
    scoring='roc_auc', n_jobs=-1,
    random_state=42, return_train_score=True
)
rs_fixed.fit(Xc_tr_sc, yc_tr)

gs_test_auc = roc_auc_score(yc_te, gs_fixed.predict_proba(Xc_te_sc)[:,1])
rs_test_auc = roc_auc_score(yc_te, rs_fixed.predict_proba(Xc_te_sc)[:,1])

print(f'Same budget ({budget} evaluations × 5 folds = {budget*5} fits):')
print(f'  GridSearchCV    : best CV AUC={gs_fixed.best_score_:.4f} | '
      f'test AUC={gs_test_auc:.4f}')
print(f'  RandomizedSearch: best CV AUC={rs_fixed.best_score_:.4f} | '
      f'test AUC={rs_test_auc:.4f}')
print(f'  Best params (Grid)  : {gs_fixed.best_params_}')
print(f'  Best params (Random): {rs_fixed.best_params_}')

# Score distribution comparison
gs_all = pd.DataFrame(gs_fixed.cv_results_)['mean_test_score'].values
rs_all = pd.DataFrame(rs_fixed.cv_results_)['mean_test_score'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(gs_all, bins=15, color=COLORS['primary'], alpha=0.75,
             label=f'Grid  (best={gs_fixed.best_score_:.3f})', edgecolor='white')
axes[0].hist(rs_all, bins=15, color=COLORS['secondary'], alpha=0.75,
             label=f'Random (best={rs_fixed.best_score_:.3f})', edgecolor='white')
axes[0].axvline(gs_fixed.best_score_, color=COLORS['primary'],
                linestyle='--', linewidth=2)
axes[0].axvline(rs_fixed.best_score_, color=COLORS['secondary'],
                linestyle='--', linewidth=2)
axes[0].set_xlabel('CV ROC-AUC', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title(f'Score Distribution — {budget} Evaluations Each',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

comp_data = {
    'Method'     : ['GridSearchCV', 'RandomizedSearch'],
    'CV AUC'     : [gs_fixed.best_score_, rs_fixed.best_score_],
    'Test AUC'   : [gs_test_auc, rs_test_auc],
}
comp_df = pd.DataFrame(comp_data)
x = np.arange(2); w = 0.35
axes[1].bar(x - w/2, comp_df['CV AUC'],   w, label='CV AUC',
            color=COLORS['primary'], alpha=0.85)
axes[1].bar(x + w/2, comp_df['Test AUC'], w, label='Test AUC',
            color=COLORS['accent'], alpha=0.85)
for i, (cv, te) in enumerate(zip(comp_df['CV AUC'], comp_df['Test AUC'])):
    axes[1].text(i-w/2, cv+0.002, f'{cv:.4f}', ha='center', fontsize=10)
    axes[1].text(i+w/2, te+0.002, f'{te:.4f}', ha='center', fontsize=10)
axes[1].set_xticks(x)
axes[1].set_xticklabels(['GridSearchCV', 'RandomizedSearch'], fontsize=11)
axes[1].set_ylabel('ROC-AUC', fontsize=11)
axes[1].set_title('Grid vs Random — Performance Comparison', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10); axes[1].set_ylim([0.7, 1.0])

plt.suptitle(f'GridSearchCV vs RandomizedSearchCV — Budget={budget} Evaluations',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6️⃣ Multi-Model Tuning — Leaderboard

> Tuning multiple models and comparing them on a unified leaderboard  
> reveals which model benefits most from tuning and which is best overall.


In [ ]:
tuning_configs = {
    'LogisticRegression': (
        LogisticRegression(max_iter=1000, random_state=42),
        {'C': [0.001, 0.01, 0.1, 1, 10, 100],
         'penalty': ['l1', 'l2'],
         'solver' : ['liblinear']},
        'grid'
    ),
    'RandomForest': (
        RandomForestClassifier(random_state=42),
        {'n_estimators': randint(50, 300),
         'max_depth'   : randint(2, 15),
         'max_features': uniform(0.2, 0.8),
         'min_samples_leaf': randint(1, 10)},
        'random'
    ),
    'GradientBoosting': (
        GradientBoostingClassifier(random_state=42),
        {'n_estimators' : randint(50, 300),
         'max_depth'    : randint(2, 8),
         'learning_rate': loguniform(0.005, 0.3),
         'subsample'    : uniform(0.5, 0.5)},
        'random'
    ),
    'SVM': (
        SVC(probability=True, random_state=42),
        {'C'    : loguniform(0.01, 100),
         'gamma': loguniform(0.001, 1.0),
         'kernel': ['rbf', 'poly']},
        'random'
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'n_neighbors': list(range(3, 31, 2)),
         'weights'    : ['uniform', 'distance'],
         'metric'     : ['euclidean', 'manhattan']},
        'grid'
    ),
}

leaderboard = []
print('Tuning Models...')
for name, (model, params, method) in tuning_configs.items():
    if method == 'grid':
        search = GridSearchCV(model, params, cv=skf, scoring='roc_auc',
                               n_jobs=-1, return_train_score=True)
    else:
        search = RandomizedSearchCV(model, params, n_iter=40, cv=skf,
                                     scoring='roc_auc', n_jobs=-1,
                                     random_state=42, return_train_score=True)
    search.fit(Xc_tr_sc, yc_tr)
    test_auc = roc_auc_score(yc_te, search.predict_proba(Xc_te_sc)[:,1])
    test_acc = accuracy_score(yc_te, search.predict(Xc_te_sc))

    leaderboard.append({
        'Model'         : name,
        'Method'        : method.capitalize(),
        'Best CV AUC'   : round(search.best_score_, 4),
        'Test AUC'      : round(test_auc, 4),
        'Test Accuracy' : round(test_acc, 4),
        'Overfit Gap'   : round(search.best_score_ - test_auc, 4),
        'Best Params'   : str(search.best_params_),
    })
    print(f'  {name:20s}: CV={search.best_score_:.4f} | Test={test_auc:.4f}')

lb_df = pd.DataFrame(leaderboard).sort_values('Test AUC', ascending=False)
print('\n📊 Tuned Model Leaderboard:')
print(lb_df[['Model','Method','Best CV AUC','Test AUC','Test Accuracy','Overfit Gap']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(lb_df)); w = 0.35
axes[0].bar(x - w/2, lb_df['Best CV AUC'], w, label='CV AUC',
            color=COLORS['primary'], alpha=0.85)
axes[0].bar(x + w/2, lb_df['Test AUC'],    w, label='Test AUC',
            color=COLORS['accent'], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(lb_df['Model'], rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('ROC-AUC', fontsize=11)
axes[0].set_title('Tuned Models — CV vs Test AUC', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].set_ylim([0.7, 1.05])

gap_colors = [COLORS['secondary'] if g > 0.02 else COLORS['accent']
              for g in lb_df['Overfit Gap']]
axes[1].bar(lb_df['Model'], lb_df['Overfit Gap'], color=gap_colors,
            alpha=0.85, edgecolor='white')
axes[1].axhline(0.02, color=COLORS['warning'], linestyle='--',
                linewidth=2, label='Overfit threshold (0.02)')
axes[1].set_xticklabels(lb_df['Model'], rotation=25, ha='right', fontsize=9)
axes[1].set_ylabel('CV AUC − Test AUC', fontsize=11)
axes[1].set_title('Overfitting Gap per Model', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Multi-Model Tuning Leaderboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7️⃣ Heatmap — 2D Parameter Interaction

> Heatmaps reveal how two hyperparameters interact —  
> which combinations are best, and whether there's a clear sweet spot.


In [ ]:
# ── Heatmap: C vs gamma for SVM ──────────────────────────────────────────
C_vals     = [0.01, 0.1, 1, 10, 100]
gamma_vals = [0.001, 0.01, 0.1, 1.0]

grid_svm = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=42),
    {'C': C_vals, 'gamma': gamma_vals},
    cv=skf, scoring='roc_auc', n_jobs=-1
)
grid_svm.fit(Xc_tr_sc, yc_tr)

scores_matrix = grid_svm.cv_results_['mean_test_score'].reshape(
    len(C_vals), len(gamma_vals)
)
scores_df = pd.DataFrame(
    scores_matrix,
    index=[f'C={c}' for c in C_vals],
    columns=[f'γ={g}' for g in gamma_vals]
)

# ── Heatmap: n_estimators vs max_depth for RF ──────────────────────────────
ne_vals = [50, 100, 150, 200, 300]
md_vals = [2, 4, 6, 8, None]
md_labels = ['2', '4', '6', '8', 'None']

grid_rf2 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'n_estimators': ne_vals, 'max_depth': md_vals},
    cv=skf, scoring='roc_auc', n_jobs=-1
)
grid_rf2.fit(Xc_tr_sc, yc_tr)

scores_rf = grid_rf2.cv_results_['mean_test_score'].reshape(
    len(ne_vals), len(md_vals)
)
scores_rf_df = pd.DataFrame(
    scores_rf,
    index=[f'n={n}' for n in ne_vals],
    columns=[f'depth={d}' for d in md_labels]
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

sns.heatmap(scores_df, annot=True, fmt='.3f', cmap='RdYlGn',
            ax=axes[0], linewidths=0.5, linecolor='white',
            vmin=scores_df.values.min(), vmax=scores_df.values.max(),
            cbar_kws={'shrink': 0.8})
axes[0].set_title('SVM — CV ROC-AUC: C vs gamma (RBF kernel)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('gamma', fontsize=11)
axes[0].set_ylabel('C', fontsize=11)

# Mark best cell
best_idx_svm = np.unravel_index(scores_matrix.argmax(), scores_matrix.shape)
axes[0].add_patch(plt.Rectangle(
    (best_idx_svm[1], best_idx_svm[0]), 1, 1,
    fill=False, edgecolor='black', linewidth=3
))

sns.heatmap(scores_rf_df, annot=True, fmt='.3f', cmap='RdYlGn',
            ax=axes[1], linewidths=0.5, linecolor='white',
            vmin=scores_rf_df.values.min(), vmax=scores_rf_df.values.max(),
            cbar_kws={'shrink': 0.8})
axes[1].set_title('Random Forest — CV ROC-AUC: n_estimators vs max_depth',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('max_depth', fontsize=11)
axes[1].set_ylabel('n_estimators', fontsize=11)

best_idx_rf = np.unravel_index(scores_rf.argmax(), scores_rf.shape)
axes[1].add_patch(plt.Rectangle(
    (best_idx_rf[1], best_idx_rf[0]), 1, 1,
    fill=False, edgecolor='black', linewidth=3
))

plt.suptitle('Hyperparameter Interaction Heatmaps', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'SVM best: C={grid_svm.best_params_["C"]}, gamma={grid_svm.best_params_["gamma"]}  '
      f'AUC={grid_svm.best_score_:.4f}')
print(f'RF  best: n={grid_rf2.best_params_["n_estimators"]}, '
      f'depth={grid_rf2.best_params_["max_depth"]}  AUC={grid_rf2.best_score_:.4f}')

---
## 8️⃣ Pipeline + GridSearchCV — Correct Leakage-Free Tuning

> ⚠️ **Critical rule:** Scaling (and any preprocessing) must be **inside**  
> the pipeline so it is fitted only on the training fold — not the test fold.
>
> Pipeline parameter names follow the pattern: `stepname__paramname`


In [ ]:
# ── Pipeline with StandardScaler + RF ────────────────────────────────────
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(random_state=42)),
])

param_grid_pipe = {
    'clf__n_estimators'    : [100, 200, 300],
    'clf__max_depth'       : [3, 5, 7, None],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features'    : ['sqrt', 'log2'],
}

gs_pipe = GridSearchCV(
    pipe_rf, param_grid_pipe,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1,
    return_train_score=True,
)

# Fit on RAW (unscaled) X — scaler handles it inside the pipeline
gs_pipe.fit(Xc_tr, yc_tr)

test_auc_pipe = roc_auc_score(
    yc_te, gs_pipe.predict_proba(Xc_te)[:,1]
)

print('Pipeline + GridSearchCV (scaler inside pipeline):')
print(f'  Best params : {gs_pipe.best_params_}')
print(f'  Best CV AUC : {gs_pipe.best_score_:.4f}')
print(f'  Test AUC    : {test_auc_pipe:.4f}')

# Show train vs val score per fold for best config
cv_pipe_df = pd.DataFrame(gs_pipe.cv_results_)
best_row    = cv_pipe_df.loc[cv_pipe_df['mean_test_score'].idxmax()]

print(f'\n  Train AUC (best config): {best_row["mean_train_score"]:.4f}')
print(f'  Val AUC   (best config): {best_row["mean_test_score"]:.4f}')
print(f'  Overfit gap            : {best_row["mean_train_score"]-best_row["mean_test_score"]:.4f}')

# Top 10 pipeline configs
top10_pipe = cv_pipe_df.nlargest(10, 'mean_test_score')[[
    'param_clf__n_estimators','param_clf__max_depth',
    'param_clf__min_samples_leaf','param_clf__max_features',
    'mean_train_score','mean_test_score','std_test_score'
]].round(4)
top10_pipe.columns = ['n_est','depth','min_leaf','max_feat',
                       'Train AUC','Val AUC','Val Std']
print('\nTop 10 Pipeline Configurations:')
print(top10_pipe.to_string(index=False))

---
## 9️⃣ Nested CV — Unbiased Evaluation After Tuning

> If you tune hyperparameters with CV then evaluate with the **same CV**,  
> the performance estimate is **optimistically biased**.
>
> **Nested CV** separates tuning (inner loop) from evaluation (outer loop):
> - Outer loop: evaluate generalization
> - Inner loop: tune hyperparameters (GridSearchCV)


In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

pipe_nested = Pipeline([
    ('sc',  StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42)),
])

param_grid_nested = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth'   : [3, 5, None],
    'clf__max_features': ['sqrt', 'log2'],
}

# Non-nested: tune once on all train, then CV
gs_nn = GridSearchCV(pipe_nested, param_grid_nested,
                      cv=inner_cv, scoring='roc_auc', n_jobs=-1)
gs_nn.fit(Xc_tr, yc_tr)
non_nested_scores = cross_val_score(
    gs_nn.best_estimator_, Xc_tr, yc_tr,
    cv=outer_cv, scoring='roc_auc'
)

# Nested: inner GridSearch inside each outer fold
gs_nested = GridSearchCV(pipe_nested, param_grid_nested,
                          cv=inner_cv, scoring='roc_auc', n_jobs=-1)
nested_scores = cross_val_score(
    gs_nested, Xc_tr, yc_tr,
    cv=outer_cv, scoring='roc_auc', n_jobs=-1
)

bias = non_nested_scores.mean() - nested_scores.mean()

print('Non-Nested CV (biased estimate):')
print(f'  Scores: {[round(s,4) for s in non_nested_scores]}')
print(f'  Mean  : {non_nested_scores.mean():.4f} ± {non_nested_scores.std():.4f}')
print()
print('Nested CV (unbiased estimate):')
print(f'  Scores: {[round(s,4) for s in nested_scores]}')
print(f'  Mean  : {nested_scores.mean():.4f} ± {nested_scores.std():.4f}')
print()
print(f'Optimism Bias (non-nested − nested): {bias:+.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1,6), non_nested_scores, 'o-', color=COLORS['warning'],
        linewidth=2.5, markersize=9,
        label=f'Non-Nested (biased)  mean={non_nested_scores.mean():.3f}')
ax.plot(range(1,6), nested_scores, 's-', color=COLORS['primary'],
        linewidth=2.5, markersize=9,
        label=f'Nested CV (unbiased) mean={nested_scores.mean():.3f}')
ax.fill_between(range(1,6),
                nested_scores.mean()-nested_scores.std(),
                nested_scores.mean()+nested_scores.std(),
                alpha=0.12, color=COLORS['primary'])
ax.set_xlabel('Outer Fold', fontsize=11)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title(f'Nested vs Non-Nested CV — Optimism Bias = {bias:+.4f}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.set_xticks(range(1,6))
plt.tight_layout()
plt.show()

---
## 🔟 Regression Hyperparameter Tuning

> GridSearchCV works identically for regression — just change `scoring`.  
> Common regression scoring: `'r2'`, `'neg_mean_squared_error'`, `'neg_mean_absolute_error'`


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Ridge Regression ──────────────────────────────────────────────────────
ridge_grid = GridSearchCV(
    Ridge(),
    {'alpha': np.logspace(-3, 4, 20)},
    cv=kf, scoring='r2', n_jobs=-1
)
ridge_grid.fit(Xr_tr_sc, yr_tr)
r2_ridge = r2_score(yr_te, ridge_grid.predict(Xr_te_sc))
print(f'Ridge: best_alpha={ridge_grid.best_params_["alpha"]:.5f} | '
      f'CV R²={ridge_grid.best_score_:.4f} | Test R²={r2_ridge:.4f}')

# ── Random Forest Regressor ────────────────────────────────────────────────
rfr_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': randint(50, 300),
     'max_depth'   : randint(2, 15),
     'max_features': uniform(0.2, 0.8),
     'min_samples_leaf': randint(1, 10)},
    n_iter=40, cv=kf, scoring='r2',
    n_jobs=-1, random_state=42
)
rfr_search.fit(Xr_tr_sc, yr_tr)
r2_rfr = r2_score(yr_te, rfr_search.predict(Xr_te_sc))
print(f'RF Reg: best_params={rfr_search.best_params_} | '
      f'CV R²={rfr_search.best_score_:.4f} | Test R²={r2_rfr:.4f}')

# Ridge alpha path
alphas_ridge = np.logspace(-3, 4, 50)
ridge_path   = GridSearchCV(
    Ridge(), {'alpha': alphas_ridge}, cv=kf, scoring='r2', n_jobs=-1
)
ridge_path.fit(Xr_tr_sc, yr_tr)
cv_r2_path = ridge_path.cv_results_['mean_test_score']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].semilogx(alphas_ridge, cv_r2_path, 'o-',
                  color=COLORS['primary'], linewidth=2, markersize=4)
axes[0].axvline(ridge_grid.best_params_['alpha'],
                color=COLORS['secondary'], linestyle='--', linewidth=2.5,
                label=f'Best alpha={ridge_grid.best_params_["alpha"]:.4f}')
axes[0].set_xlabel('Alpha (log scale)', fontsize=11)
axes[0].set_ylabel('CV R²', fontsize=11)
axes[0].set_title('Ridge — CV R² vs Alpha', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# RF vs Ridge comparison
reg_models = ['Ridge
(tuned)', 'RF Regressor
(tuned)']
test_r2s   = [r2_ridge, r2_rfr]
cv_r2s     = [ridge_grid.best_score_, rfr_search.best_score_]
x = np.arange(2); w = 0.35
axes[1].bar(x-w/2, cv_r2s,   w, label='CV R²',   color=COLORS['primary'],   alpha=0.85)
axes[1].bar(x+w/2, test_r2s, w, label='Test R²', color=COLORS['accent'],    alpha=0.85)
for i, (cv, te) in enumerate(zip(cv_r2s, test_r2s)):
    axes[1].text(i-w/2, cv+0.005, f'{cv:.3f}', ha='center', fontsize=11)
    axes[1].text(i+w/2, te+0.005, f'{te:.3f}', ha='center', fontsize=11)
axes[1].set_xticks(x); axes[1].set_xticklabels(reg_models, fontsize=11)
axes[1].set_ylabel('R² Score', fontsize=11)
axes[1].set_title('Regression Tuning — CV vs Test R²', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10); axes[1].set_ylim([0.5, 1.05])

plt.suptitle('Regression Hyperparameter Tuning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ✅ 11. Summary & Golden Rules

| Method | When to Use | Handles Distributions? | sklearn Class |
|--------|-------------|:---------------------:|---------------|
| **GridSearchCV** | Small search space (≤ 3 params, ≤ 5 values each) | ❌ Lists only | `GridSearchCV` |
| **RandomizedSearchCV** | Large search space, limited budget | ✅ scipy distributions | `RandomizedSearchCV` |
| **Bayesian Optimization** | Very expensive evaluations (deep learning) | ✅ Yes | `skopt`, `Optuna` |

### 🔑 Golden Rules

1. **Always establish a baseline** with default params before tuning
2. **Always use a Pipeline** — scaler must be inside CV folds, not outside
3. **Use `scoring='roc_auc'`** for classification, `'r2'` or `'neg_rmse'` for regression
4. **Random Search ≥ Grid Search** for spaces with > 3 hyperparameters
5. **Use log-uniform distributions** for learning rates, C, alpha, gamma
6. **Use `return_train_score=True`** to detect overfitting during tuning
7. **Nested CV gives an unbiased estimate** — use it when reporting final performance
8. **Plot heatmaps** for 2-param interactions — they reveal the search landscape
9. **n_iter=40–60** is usually enough for RandomizedSearchCV on most models
10. **Check overfit gap** (train − val score) after tuning — tuning can cause overfitting too

---

## 🔗 Next Steps

- ➡️ `07_Hyperparameter_Tuning/bayesian_optimization.md` — More efficient search
- ➡️ `05_Model_Evaluation/cross_validation.ipynb` — CV strategies for evaluation
- ➡️ `08_Ensemble_Learning/` — Tune ensemble hyperparameters
- ➡️ `XGBoost`, `LightGBM`, `CatBoost` — Primary candidates for tuning
